# Create vector labels for training deep learning models

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/create_vector.ipynb)

## Install package
To use the `geoai-py` package, ensure it is installed in your environment. Uncomment the command below if needed.

In [5]:
# %pip install geoai-py
%pip install pymupdf pillow rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 67.2 MB/s eta 0:00:00


## Import libraries

In [6]:
import geoai
import fitz  # PyMuPDF
from PIL import Image
import numpy as np
import rasterio
from rasterio.transform import from_origin


In [ ]:
pdf_path = "/content/test.pdf"
output_tif = "output.tif"

# Open PDF
doc = fitz.open(pdf_path)

# First page
page = doc[0]

# Render page to image
pix = page.get_pixmap(matrix=fitz.Matrix(4, 4))  # higher resolution

# Convert to PIL image
img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

# Convert to numpy array
arr = np.array(img)

# Dummy transform (no real coordinates)
transform = from_origin(0, 0, 1, 1)

# Save as GeoTIFF
with rasterio.open(
    output_tif,
    "w",
    driver="GTiff",
    height=arr.shape[0],
    width=arr.shape[1],
    count=3,
    dtype=arr.dtype,
    crs="EPSG:4326",  # fake CRS if needed
    transform=transform,
) as dst:
    for i in range(3):
        dst.write(arr[:, :, i], i + 1)

print("Saved:", output_tif)

## Add sample datasets

In [4]:
m = geoai.Map(style="liberty")
raster_url = (
    "/content/test.pdf"
)
m.add_cog_layer(raster_url, name="NAIP")
m.add_layer_control()
m.add_draw_control(
    controls=["point", "polygon", "line_string", "trash"], position="top-right"
)

'band_descriptions'


RasterioIOError: '/content/test.pdf' not recognized as being in a supported file format.

## Set default properties

In [ ]:
properties = {
    "Type": ["Residential", "Commercial", "Industrial"],
    "Area": 3000,
    "Name": "Building",
    "City": "Seattle",
}

## Display the interactive widget

In [ ]:
widget = geoai.create_vector_data(m, properties, file_ext="gpkg")
widget

![image](https://github.com/user-attachments/assets/822fced3-5ba9-407c-9a17-e9332e2d84ad)